# 🐺 Whispers of the Seven Kingdoms – MusicGen Studio v2

Generiere 40+ Minuten Sleep Music im Game of Thrones-Stil.

**Setup:** Runtime → Change runtime type → **T4 GPU** auswählen!

Dann einfach alle Zellen von oben nach unten ausführen.

In [ ]:
# 1. Installation
!pip install -q transformers torch scipy torchaudio pydub
!apt-get -qq install ffmpeg

In [ ]:
# 2. Modell laden
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import torch
import scipy.io.wavfile
import numpy as np
from IPython.display import Audio, display
from pydub import AudioSegment
import os, random, time

MODEL_NAME = 'facebook/musicgen-medium'

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = MusicgenForConditionalGeneration.from_pretrained(MODEL_NAME)
model = model.to('cuda')

SAMPLE_RATE = model.config.audio_encoder.sampling_rate
OUTPUT_DIR = '/content/generated_tracks'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'✅ {MODEL_NAME} geladen auf GPU!')
print(f'📁 Output: {OUTPUT_DIR}')

---
## ⚙️ Konfiguration

Hier stellst du ein **was** und **wie lange** generiert wird.

In [ ]:
# 3. KONFIGURATION

# === ZIEL-LÄNGE ===
TARGET_MINUTES = 42  # Ziel: etwas über 40 Min
CLIP_SECONDS = 30    # Länge pro Clip (max 30 empfohlen)
CROSSFADE_SEC = 4    # Crossfade zwischen Clips

# === TRACK-NAME (= Slug für Pipeline) ===
TRACK_NAME = "whispers-of-winterfell"

# === PROMPTS ===
PROMPTS = [
    "Gentle ambient sleep music inspired by a frozen northern castle. Soft piano melody over subtle string pads with cold wind textures. Calm, meditative, melancholic. No vocals, no drums. Tempo 50 BPM, minor key. Evolving pads that shift every 8 bars.",
    "Soft piano melody, winter night, gentle wind, peaceful and warm, medieval castle, sleep music, no vocals, no drums. Tempo 50 BPM, minor key.",
    "Gentle harp and piano, quiet snowfall, ancient castle ambience, calm and serene, deep sleep, no vocals, no drums. Tempo 50 BPM, minor key.",
    "Slow peaceful piano, crackling fireplace, warm medieval hall, soothing lullaby, sleep ambient, no vocals, no drums. Tempo 50 BPM, minor key.",
    "Soft strings and piano, cold winter night, gentle and melancholic, peaceful castle, sleep music, no vocals, no drums. Tempo 50 BPM, minor key.",
]

# === UPLOAD-ZIEL ===
UPLOAD_TARGET = "github"

# GitHub Config
GITHUB_TOKEN = ""  # Dein GitHub PAT hier eintragen!
GITHUB_REPO = "pepler1993-dot/WhispersoftheSevenKingdoms"
GITHUB_BRANCH = "main"

# === BERECHNUNG ===
effective_clip = CLIP_SECONDS - CROSSFADE_SEC
num_clips = int(np.ceil((TARGET_MINUTES * 60) / effective_clip))
estimated_minutes = (num_clips * effective_clip + CROSSFADE_SEC) / 60

print(f"🎵 Track: {TRACK_NAME}")
print(f"⏱️ Ziel: {TARGET_MINUTES} Min → {num_clips} Clips à {CLIP_SECONDS}s")
print(f"📊 Geschätzte Gesamtlänge: {estimated_minutes:.1f} Min")
print(f"🔁 {len(PROMPTS)} Prompt-Variationen")
print(f"📤 Upload: {UPLOAD_TARGET}")


---
## 🎹 Generierung starten

Generiert automatisch alle Clips und fügt sie zusammen.

In [ ]:
# 4. CLIPS GENERIEREN (Loop)

clip_dir = f'{OUTPUT_DIR}/{TRACK_NAME}_clips'
os.makedirs(clip_dir, exist_ok=True)

print(f'🎹 Generiere {num_clips} Clips...\n')
start_time = time.time()

for i in range(num_clips):
    # Prompt rotieren
    prompt = PROMPTS[i % len(PROMPTS)]

    # Leichte Zufalls-Variation für natürlicheren Sound
    temp = round(random.uniform(0.9, 1.1), 2)
    guidance = round(random.uniform(2.5, 3.5), 1)

    print(f'  [{i+1}/{num_clips}] Prompt {(i % len(PROMPTS))+1} | temp={temp} cfg={guidance}', end='... ')

    inputs = processor(text=[prompt], padding=True, return_tensors='pt').to('cuda')
    max_tokens = int(CLIP_SECONDS * 50)

    with torch.no_grad():
        audio_values = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temp,
            guidance_scale=guidance
        )

    audio_data = audio_values[0, 0].cpu().numpy()
    clip_path = f'{clip_dir}/clip_{i:03d}.wav'
    scipy.io.wavfile.write(clip_path, rate=SAMPLE_RATE, data=audio_data)

    elapsed = time.time() - start_time
    eta = (elapsed / (i+1)) * (num_clips - i - 1)
    print(f'✅ ({len(audio_data)/SAMPLE_RATE:.0f}s) ETA: {eta/60:.0f}min')

elapsed_total = (time.time() - start_time) / 60
print(f'\n🎉 Alle {num_clips} Clips generiert in {elapsed_total:.1f} Minuten!')

In [ ]:
# 5. CLIPS ZUSAMMENFÜGEN MIT CROSSFADE
import glob

def merge_clips_with_crossfade(clip_dir, output_path, crossfade_ms=4000):
    """Fügt alle WAV-Clips mit sanftem Crossfade zusammen."""
    wav_files = sorted(glob.glob(f'{clip_dir}/clip_*.wav'))
    print(f'🔗 Füge {len(wav_files)} Clips zusammen...')

    # Erster Clip
    merged = AudioSegment.from_wav(wav_files[0])

    for i, wav_file in enumerate(wav_files[1:], 1):
        clip = AudioSegment.from_wav(wav_file)
        merged = merged.append(clip, crossfade=crossfade_ms)
        if i % 20 == 0:
            print(f'  ... {i}/{len(wav_files)-1} Clips zusammengefügt')

    # Normalisieren
    merged = merged.normalize()

    # Fade-in/Fade-out am Anfang/Ende
    merged = merged.fade_in(3000).fade_out(5000)

    # Als WAV speichern
    merged.export(output_path, format='wav')
    duration_min = len(merged) / 1000 / 60
    size_mb = os.path.getsize(output_path) / 1024 / 1024

    print(f'✅ Fertig: {output_path}')
    print(f'⏱️ Länge: {duration_min:.1f} Minuten')
    print(f'💾 Größe: {size_mb:.0f} MB (WAV)')
    return output_path, duration_min

wav_path, duration = merge_clips_with_crossfade(
    clip_dir,
    f'{OUTPUT_DIR}/{TRACK_NAME}.wav',
    crossfade_ms=CROSSFADE_SEC * 1000
)

# Vorschau (erste 30 Sekunden)
preview = AudioSegment.from_wav(wav_path)[:30000]
display(Audio(preview.raw_data, rate=preview.frame_rate))

In [ ]:
# 6. ZU MP3 KONVERTIEREN (für Upload)

mp3_path = f'{OUTPUT_DIR}/{TRACK_NAME}.mp3'
audio = AudioSegment.from_wav(wav_path)
audio.export(mp3_path, format='mp3', bitrate='192k',
             tags={'artist': 'Whispers of the Seven Kingdoms',
                   'album': 'Whispers of the Seven Kingdoms',
                   'title': TRACK_NAME.replace('_', ' ').title()})

mp3_size = os.path.getsize(mp3_path) / 1024 / 1024
print(f'✅ MP3 erstellt: {mp3_path}')
print(f'💾 Größe: {mp3_size:.1f} MB')

---
## 📤 Upload

In [ ]:
# 7. AUTO-UPLOAD
import base64, requests, shutil

if UPLOAD_TARGET == 'github' and GITHUB_TOKEN:
    # === GITHUB UPLOAD ===
    print('📤 Uploade zu GitHub...')

    with open(mp3_path, 'rb') as f:
        content_b64 = base64.b64encode(f.read()).decode()

    upload_path = f'upload/songs/{TRACK_NAME}.mp3'
    api_url = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{upload_path}'

    r = requests.put(api_url, headers={
        'Authorization': f'Bearer {GITHUB_TOKEN}',
        'Accept': 'application/vnd.github.v3+json'
    }, json={
        'message': f'Add generated track: {TRACK_NAME}',
        'content': content_b64,
        'branch': GITHUB_BRANCH
    })

    if r.status_code in [200, 201]:
        print(f'✅ Gepusht: {upload_path}')
        print(f'🔗 {r.json()["content"]["html_url"]}')
    else:
        print(f'❌ Fehler: {r.status_code} – {r.text[:200]}')

elif UPLOAD_TARGET == 'nextcloud' and NEXTCLOUD_URL:
    # === NEXTCLOUD UPLOAD (WebDAV) ===
    print('📤 Uploade zu Nextcloud...')

    webdav_url = f'{NEXTCLOUD_URL}/remote.php/dav/files/{NEXTCLOUD_USER}{NEXTCLOUD_PATH}{TRACK_NAME}.mp3'

    with open(mp3_path, 'rb') as f:
        r = requests.put(webdav_url, data=f,
                        auth=(NEXTCLOUD_USER, NEXTCLOUD_PASS),
                        headers={'Content-Type': 'audio/mpeg'})

    if r.status_code in [200, 201, 204]:
        print(f'✅ Hochgeladen: {NEXTCLOUD_PATH}{TRACK_NAME}.mp3')
    else:
        print(f'❌ Fehler: {r.status_code} – {r.text[:200]}')

else:
    # === LOKALER DOWNLOAD ===
    print('📥 Lokaler Download...')
    from google.colab import files
    files.download(mp3_path)
    print('✅ Download gestartet!')

---
## 🔁 Batch: Mehrere Tracks nacheinander

Definiere eine Liste von Tracks und generiere alle automatisch.

In [ ]:
# 8. BATCH – MEHRERE TRACKS

BATCH_TRACKS = [
    {
        'name': 'winterfell_peaceful_piano',
        'prompts': [
            'peaceful medieval piano, gentle snowfall ambience, calm sleep music, soft and soothing, castle atmosphere, no vocals',
            'soft piano melody, winter night, gentle wind, peaceful and warm, medieval castle, sleep music, no vocals',
            'gentle harp and piano, quiet snowfall, ancient castle ambience, calm and serene, deep sleep, no vocals',
        ]
    },
    {
        'name': 'kings_landing_night',
        'prompts': [
            'warm ambient music, gentle harp and soft strings, mediterranean night, royal palace, calm and regal, sleep music, no vocals',
            'soft lute melody, warm evening breeze, ancient city ambience, peaceful and elegant, no vocals',
            'gentle strings and flute, candlelit palace, warm night atmosphere, soothing royal music, no vocals',
        ]
    },
    {
        'name': 'targaryen_ethereal',
        'prompts': [
            'ethereal ambient music, mysterious and ancient, soft choir pads, dark but peaceful, deep sleep music, no vocals',
            'haunting ambient, dragon inspired, deep drones, mystical atmosphere, calm and vast, no vocals',
            'ethereal strings, ancient power, slow and meditative, dark ambient sleep music, no vocals',
        ]
    },
    {
        'name': 'the_wall_dark_ambient',
        'prompts': [
            'dark ambient soundscape, cold wind, ice and snow, mysterious and vast, deep bass drones, haunting, no vocals',
            'frozen ambient, howling wind, deep cold atmosphere, dark and peaceful, sleep ambient, no vocals',
            'icy drone ambient, vast frozen landscape, slow dark pads, deep meditation, no vocals',
        ]
    },
    {
        'name': 'highgarden_gentle',
        'prompts': [
            'gentle acoustic guitar, warm summer garden, birds singing softly, peaceful, relaxing sleep music, medieval, no vocals',
            'soft flute and harp, sunny garden ambience, butterflies, warm and light, soothing, no vocals',
            'peaceful lute melody, gentle breeze, flower garden, warm afternoon, calming, no vocals',
        ]
    },
    {
        'name': 'godswood_meditation',
        'prompts': [
            'meditative ambient, ancient forest, gentle wind through leaves, mystical and sacred, soft flute, no vocals',
            'forest ambient, soft rain on leaves, ancient trees, deep calm, nature meditation, no vocals',
            'sacred ambient music, old growth forest, gentle streams, peaceful and still, spiritual, no vocals',
        ]
    },
    {
        'name': 'rains_of_castamere_sleep',
        'prompts': [
            'slow melancholic piano, gentle and sad, medieval castle, soft rain background, emotional sleep music, no vocals',
            'bittersweet piano melody, rainy night, stone castle, lonely and peaceful, sleep ambient, no vocals',
            'gentle sad piano, quiet rainfall, candle-lit hall, melancholic but calming, deep sleep, no vocals',
        ]
    },
    {
        'name': 'dorne_desert_night',
        'prompts': [
            'warm desert night ambient, soft oud, middle eastern inspired, peaceful oasis, calm sleep music, no vocals',
            'gentle arabic flute, warm night breeze, desert stars, soothing and exotic, sleep ambient, no vocals',
            'soft percussion and strings, desert oasis, warm sand, peaceful night, mediterranean calm, no vocals',
        ]
    },
]

# === BATCH STARTEN ===
print(f'🎵 Batch-Generierung: {len(BATCH_TRACKS)} Tracks à ~{TARGET_MINUTES} Min\n')

for t_idx, track in enumerate(BATCH_TRACKS):
    print(f'\n{"="*60}')
    print(f'🎹 [{t_idx+1}/{len(BATCH_TRACKS)}] {track["name"]}')
    print(f'{"="*60}')

    t_clip_dir = f'{OUTPUT_DIR}/{track["name"]}_clips'
    os.makedirs(t_clip_dir, exist_ok=True)

    for i in range(num_clips):
        prompt = track['prompts'][i % len(track['prompts'])]
        temp = round(random.uniform(0.9, 1.1), 2)
        guidance = round(random.uniform(2.5, 3.5), 1)

        if i % 10 == 0:
            print(f'  Clip {i+1}/{num_clips}...')

        inputs = processor(text=[prompt], padding=True, return_tensors='pt').to('cuda')
        max_tokens = int(CLIP_SECONDS * 50)

        with torch.no_grad():
            audio_values = model.generate(**inputs, max_new_tokens=max_tokens,
                                         temperature=temp, guidance_scale=guidance)

        audio_data = audio_values[0, 0].cpu().numpy()
        scipy.io.wavfile.write(f'{t_clip_dir}/clip_{i:03d}.wav', rate=SAMPLE_RATE, data=audio_data)

    # Zusammenfügen
    wav_path, dur = merge_clips_with_crossfade(t_clip_dir, f'{OUTPUT_DIR}/{track["name"]}.wav',
                                              crossfade_ms=CROSSFADE_SEC*1000)

    # MP3
    mp3_out = f'{OUTPUT_DIR}/{track["name"]}.mp3'
    AudioSegment.from_wav(wav_path).export(mp3_out, format='mp3', bitrate='192k',
        tags={'artist': 'Whispers of the Seven Kingdoms', 'title': track['name'].replace('_',' ').title()})

    mp3_size = os.path.getsize(mp3_out) / 1024 / 1024
    print(f'  ✅ {track["name"]}.mp3 ({dur:.0f} Min, {mp3_size:.0f} MB)')

print(f'\n🎉 Alle {len(BATCH_TRACKS)} Tracks fertig!')

In [ ]:
# 9. ALLE TRACKS DOWNLOADEN / UPLOADEN
import shutil

# Nur MP3s in ZIP packen (nicht die einzelnen Clips)
mp3_dir = f'{OUTPUT_DIR}/mp3_export'
os.makedirs(mp3_dir, exist_ok=True)
for f in glob.glob(f'{OUTPUT_DIR}/*.mp3'):
    shutil.copy2(f, mp3_dir)

shutil.make_archive('/content/whispers_all_tracks', 'zip', mp3_dir)
zip_size = os.path.getsize('/content/whispers_all_tracks.zip') / 1024 / 1024
print(f'📦 ZIP: {zip_size:.0f} MB')

from google.colab import files
files.download('/content/whispers_all_tracks.zip')

---
## 💡 Prompt-Tipps

**Gute Ergebnisse:**
- Instrumente: `piano`, `harp`, `flute`, `strings`, `guitar`, `oud`, `lute`
- Stimmung: `peaceful`, `calm`, `gentle`, `haunting`, `ethereal`, `melancholic`
- Atmosphäre: `snowfall`, `rain`, `fireplace`, `wind`, `forest`, `desert night`
- Immer `no vocals` am Ende
- Tempo: `slow`, `gentle`, `soft`

**Vermeide:**
- Eigennamen ('Winterfell') → beschreibe die Stimmung stattdessen
- Zu viele Details auf einmal

**Loop-Tipps:**
- 3-5 Prompt-Variationen pro Track halten es interessant
- Leichte Temperatur-Variation (0.9-1.1) sorgt für natürliche Unterschiede
- Crossfade von 3-5 Sek macht Übergänge unhörbar